In [1]:
import os
from urllib.request import urlretrieve
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server(
    '/tmp/hw7_pg',
    cleanup_mode='stop'
)

engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

PostgreSQL 16.2 on aarch64-apple-darwin23.5.0, compiled by Apple clang version 15.0.0 (clang-1500.0.40.1), 64-bit


In [2]:
URL = (
    'https://d37ci6vzurychx.cloudfront.net/trip-data/'
    'yellow_tripdata_2024-03.parquet'
)

os.makedirs('data', exist_ok=True)
parquet_path = 'data/yellow_tripdata_2024-03.parquet'

urlretrieve(URL, parquet_path)

columns = [
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'trip_distance',
    'fare_amount',
    'passenger_count',
]


df_all = pd.read_parquet(
    parquet_path,
    columns=columns
)


df = df_all.sample(
    n=min(500_000, len(df_all)),
    random_state=42
).copy()

del df_all


df = df[
    (df['fare_amount'] > 0) &
    (df['fare_amount'] < 500) &
    (df['trip_distance'] > 0) &
    (df['trip_distance'] < 100) &
    (df['PULocationID'].notna()) &
    (df['tpep_pickup_datetime'].notna())
]


df.to_sql(
    'nyc_taxi_raw',
    engine,
    if_exists='replace',
    index=False,
    chunksize=10_000
)

print("Rows loaded:", len(df))
print("Min date:", df['tpep_pickup_datetime'].min())
print("Max date:", df['tpep_pickup_datetime'].max())

pd.read_sql(
    'SELECT COUNT(*) AS n_rows FROM nyc_taxi_raw',
    engine
)

Rows loaded: 480035
Min date: 2024-02-29 23:37:21
Max date: 2024-04-01 00:34:55


,n_rows
0,480035


In [4]:
with engine.begin() as conn:

    conn.execute(text("""
        TRUNCATE TABLE t7_hw_taxi_daily;
    """))

    conn.execute(text("""
        INSERT INTO t7_hw_taxi_daily (
            pickup_zone_id,
            ts_day,
            trip_count,
            avg_fare,
            total_distance
        )

        SELECT
            "PULocationID"::INTEGER,
            DATE_TRUNC(
                'day',
                tpep_pickup_datetime::TIMESTAMP
            )::DATE,
            COUNT(*)::INTEGER,
            AVG(fare_amount)::NUMERIC(10, 2),
            SUM(trip_distance)::NUMERIC(12, 2)

        FROM nyc_taxi_raw

        WHERE fare_amount > 0
          AND fare_amount < 500
          AND trip_distance > 0
          AND trip_distance < 100
          AND "PULocationID" IS NOT NULL

        GROUP BY
            "PULocationID"::INTEGER,
            DATE_TRUNC(
                'day',
                tpep_pickup_datetime::TIMESTAMP
            )::DATE;
    """))

print("Daily aggregation updated!")

pd.read_sql(
    """
    SELECT
        pickup_zone_id,
        COUNT(*) AS n_days
    FROM t7_hw_taxi_daily
    GROUP BY pickup_zone_id
    ORDER BY pickup_zone_id
    LIMIT 10;
    """,
    engine
)

Daily aggregation updated!


,pickup_zone_id,n_days
0,1,14
1,3,12
2,4,31
3,6,3
4,7,31
5,9,9
6,10,31
7,11,14
8,12,31
9,13,31


In [5]:
with engine.begin() as conn:

    conn.execute(text("""
        DROP TABLE IF EXISTS t7_hw_taxi_trips_typed;
    """))

    conn.execute(text("""
        CREATE TABLE t7_hw_taxi_trips_typed AS
        SELECT
            "PULocationID"::INTEGER AS pickup_zone_id,

            "DOLocationID"::INTEGER AS dropoff_zone_id,

            (
                tpep_pickup_datetime::TIMESTAMP
                AT TIME ZONE 'America/New_York'
            ) AS pickup_at,

            (
                tpep_dropoff_datetime::TIMESTAMP
                AT TIME ZONE 'America/New_York'
            ) AS dropoff_at,

            trip_distance::NUMERIC(10, 2)
                AS trip_distance,

            fare_amount::NUMERIC(10, 2)
                AS fare_amount,

            passenger_count::INTEGER
                AS passenger_count

        FROM nyc_taxi_raw

        WHERE fare_amount > 0
          AND fare_amount < 500
          AND trip_distance > 0
          AND trip_distance < 100;
    """))

# Перевірка типу TIMESTAMPTZ
pd.read_sql(
    """
    SELECT
        pickup_at,
        pg_typeof(pickup_at) AS pickup_type
    FROM t7_hw_taxi_trips_typed
    LIMIT 5;
    """,
    engine
)

,pickup_at,pickup_type
0,2024-03-28 00:26:56+00:00,timestamp with time zone
1,2024-03-25 02:49:55+00:00,timestamp with time zone
2,2024-03-10 14:21:00+00:00,timestamp with time zone
3,2024-03-26 12:31:20+00:00,timestamp with time zone
4,2024-03-23 17:01:35+00:00,timestamp with time zone


In [6]:
pd.read_sql(
    """
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT pickup_zone_id) AS total_zones,
        MIN(ts_day) AS min_date,
        MAX(ts_day) AS max_date,
        SUM(trip_count) AS total_trips
    FROM t7_hw_taxi_daily;
    """,
    engine
)

,total_rows,total_zones,min_date,max_date,total_trips
0,5420,245,2024-02-29,2024-04-01,480035


In [7]:
pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT pickup_zone_id) AS total_zones,
        COUNT(DISTINCT ts_day) AS total_days,
        MIN(ts_day) AS min_date,
        MAX(ts_day) AS max_date
    FROM t7_hw_taxi_daily;
""", engine)

,total_rows,total_zones,total_days,min_date,max_date
0,5420,245,33,2024-02-29,2024-04-01


In [8]:
df = pd.read_parquet(
    parquet_path,
    columns=columns
).head(500_000).copy()

In [9]:
df = pd.read_parquet(
    parquet_path,
    columns=columns
).sample(n=500_000, random_state=42).copy()

In [10]:
query_lag = """
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,

    LAG(trip_count, 1) OVER w AS trips_lag_1d,
    LAG(trip_count, 7) OVER w AS trips_lag_7d,
    LAG(trip_count, 30) OVER w AS trips_lag_30d

FROM t7_hw_taxi_daily

WINDOW w AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
)

ORDER BY pickup_zone_id, ts_day
LIMIT 20;
"""

pd.read_sql(query_lag, engine)

,pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
0,1,2024-03-04,1,NaN,NaN,None
1,1,2024-03-08,1,1.0,NaN,None
2,1,2024-03-10,2,1.0,NaN,None
3,1,2024-03-11,1,2.0,NaN,None
4,1,2024-03-14,1,1.0,NaN,None
5,1,2024-03-18,1,1.0,NaN,None
6,1,2024-03-21,2,1.0,NaN,None
7,1,2024-03-22,1,2.0,1.0,None
8,1,2024-03-23,1,1.0,1.0,None
9,1,2024-03-24,1,1.0,2.0,None


In [11]:
query_rolling_7 = """
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,

    AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,

    COUNT(*) OVER w7 AS days_in_window

FROM t7_hw_taxi_daily

WINDOW w7 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP

    RANGE BETWEEN
        INTERVAL '6 days' PRECEDING
        AND CURRENT ROW
)

ORDER BY pickup_zone_id, ts_day
LIMIT 20;
"""

pd.read_sql(query_rolling_7, engine)

,pickup_zone_id,ts_day,avg_fare,rolling_avg_fare_7d,days_in_window
0,1,2024-03-04,40.00,40.000000,1
1,1,2024-03-08,105.00,72.500000,2
2,1,2024-03-10,23.00,56.000000,3
3,1,2024-03-11,3.00,43.666667,3
4,1,2024-03-14,87.40,54.600000,4
5,1,2024-03-18,128.30,107.850000,2
6,1,2024-03-21,86.55,107.425000,2
7,1,2024-03-22,82.80,99.216667,3
8,1,2024-03-23,105.00,100.662500,4
9,1,2024-03-24,35.00,87.530000,5


In [12]:
query_rolling_30 = """
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,

    STDDEV_SAMP(avg_fare) OVER w30
        AS rolling_std_fare_30d,

    COUNT(*) OVER w30 AS days_in_window

FROM t7_hw_taxi_daily

WINDOW w30 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP

    RANGE BETWEEN
        INTERVAL '29 days' PRECEDING
        AND CURRENT ROW
)

ORDER BY pickup_zone_id, ts_day
LIMIT 20;
"""

pd.read_sql(query_rolling_30, engine)

,pickup_zone_id,ts_day,avg_fare,rolling_std_fare_30d,days_in_window
0,1,2024-03-04,40.00,NaN,1
1,1,2024-03-08,105.00,45.961941,2
2,1,2024-03-10,23.00,43.278170,3
3,1,2024-03-11,3.00,44.169182,4
4,1,2024-03-14,87.40,43.149878,5
5,1,2024-03-18,128.30,49.678637,6
6,1,2024-03-21,86.55,46.113035,7
7,1,2024-03-22,82.80,43.028964,8
8,1,2024-03-23,105.00,41.952756,9
9,1,2024-03-24,35.00,41.380124,10


In [13]:
query_expanding = """
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,

    SUM(trip_count) OVER we AS cumulative_trips,

    AVG(trip_count) OVER we AS expanding_avg_trips

FROM t7_hw_taxi_daily

WINDOW we AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day

    ROWS BETWEEN
        UNBOUNDED PRECEDING
        AND CURRENT ROW
)

ORDER BY pickup_zone_id, ts_day
LIMIT 20;
"""

pd.read_sql(query_expanding, engine)

,pickup_zone_id,ts_day,trip_count,cumulative_trips,expanding_avg_trips
0,1,2024-03-04,1,1,1.000000
1,1,2024-03-08,1,2,1.000000
2,1,2024-03-10,2,4,1.333333
3,1,2024-03-11,1,5,1.250000
4,1,2024-03-14,1,6,1.200000
5,1,2024-03-18,1,7,1.166667
6,1,2024-03-21,2,9,1.285714
7,1,2024-03-22,1,10,1.250000
8,1,2024-03-23,1,11,1.222222
9,1,2024-03-24,1,12,1.200000


In [14]:
create_view_query = """
CREATE OR REPLACE VIEW v_t7_hw_taxi_features AS

WITH features_basic AS (

    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        avg_fare,
        total_distance,

        -- Feature 1: Lag features
        LAG(trip_count, 1) OVER w_pid
            AS trips_lag_1d,

        LAG(trip_count, 7) OVER w_pid
            AS trips_lag_7d,

        LAG(trip_count, 30) OVER w_pid
            AS trips_lag_30d,

        -- Feature 2: Rolling 7-day average
        AVG(avg_fare) OVER w7
            AS rolling_avg_fare_7d,

        -- Feature 3: Rolling 30-day volatility
        STDDEV_SAMP(avg_fare) OVER w30
            AS rolling_std_fare_30d,

        -- Feature 4: Expanding features
        SUM(trip_count) OVER we
            AS cumulative_trips,

        AVG(trip_count) OVER we
            AS expanding_avg_trips

    FROM t7_hw_taxi_daily

    WINDOW

        w_pid AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
        ),

        w7 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP

            RANGE BETWEEN
                INTERVAL '6 days' PRECEDING
                AND CURRENT ROW
        ),

        w30 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP

            RANGE BETWEEN
                INTERVAL '29 days' PRECEDING
                AND CURRENT ROW
        ),

        we AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day

            ROWS BETWEEN
                UNBOUNDED PRECEDING
                AND CURRENT ROW
        )
)

SELECT *
FROM features_basic
WHERE trips_lag_30d IS NOT NULL;
"""

with engine.begin() as conn:
    conn.execute(text(create_view_query))

print("View v_t7_hw_taxi_features created successfully!")

View v_t7_hw_taxi_features created successfully!


In [15]:
df_features = pd.read_sql("""
    SELECT *
    FROM v_t7_hw_taxi_features
    ORDER BY pickup_zone_id, ts_day
    LIMIT 20;
""", engine)

display(df_features)

,pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,trips_lag_1d,trips_lag_7d,trips_lag_30d,rolling_avg_fare_7d,rolling_std_fare_30d,cumulative_trips,expanding_avg_trips
0,4,2024-03-31,63,16.93,181.13,65,68,25,17.137143,2.489348,995,32.096774
1,7,2024-03-31,16,18.03,48.15,12,12,13,19.782857,4.538983,306,9.870968
2,10,2024-03-31,5,42.87,49.98,1,6,5,50.695714,14.198069,163,5.258065
3,12,2024-03-31,6,33.22,45.02,11,5,4,25.910000,5.876572,152,4.903226
4,13,2024-03-31,45,22.31,201.07,76,56,90,21.604286,2.200139,2786,89.870968
5,17,2024-03-31,4,36.70,33.67,5,4,3,25.807143,9.644677,132,4.258065
6,24,2024-03-31,33,15.17,88.44,31,34,46,15.781429,2.040569,1204,38.838710
7,33,2024-03-31,5,33.38,35.79,7,12,9,28.792857,5.385944,294,9.483871
8,35,2024-03-31,4,42.25,41.94,2,5,11,31.970000,5.885600,125,4.032258
9,39,2024-03-31,6,27.85,39.32,4,2,2,31.244286,10.348544,128,4.129032


In [16]:
pd.read_sql("""
    SELECT
        COUNT(*) AS total_feature_rows,
        COUNT(DISTINCT pickup_zone_id) AS total_zones,

        COUNT(trips_lag_1d) AS lag_1d_count,
        COUNT(trips_lag_7d) AS lag_7d_count,
        COUNT(trips_lag_30d) AS lag_30d_count,

        COUNT(rolling_avg_fare_7d) AS rolling_7d_count,
        COUNT(rolling_std_fare_30d) AS rolling_30d_count

    FROM v_t7_hw_taxi_features;
""", engine)

,total_feature_rows,total_zones,lag_1d_count,lag_7d_count,lag_30d_count,rolling_7d_count,rolling_30d_count
0,87,83,87,87,87,87,87


In [17]:
pd.read_sql("""
    SELECT
        MIN(ts_day) AS first_day,
        MAX(ts_day) AS last_day,
        COUNT(DISTINCT ts_day) AS total_days
    FROM t7_hw_taxi_daily;
""", engine)

,first_day,last_day,total_days
0,2024-02-29,2024-04-01,33


JSONB-таблиця для подій

In [18]:
with engine.begin() as conn:

    conn.execute(text("""
        DROP TABLE IF EXISTS t7_hw_events;
    """))

    conn.execute(text("""
        CREATE TABLE t7_hw_events (
            event_id INTEGER GENERATED BY DEFAULT
                AS IDENTITY PRIMARY KEY,

            user_id INTEGER NOT NULL,

            event_ts TIMESTAMPTZ NOT NULL
                DEFAULT NOW(),

            event_type TEXT NOT NULL,

            payload JSONB NOT NULL
        );
    """))

print("Table t7_hw_events created successfully!")

Table t7_hw_events created successfully!


In [19]:
insert_query = """
INSERT INTO t7_hw_events (
    user_id,
    event_ts,
    event_type,
    payload
)

SELECT
    (n % 1000) + 1 AS user_id,

    TIMESTAMPTZ '2024-04-01 00:00:00+00'
        - ((n % 60) || ' days')::INTERVAL AS event_ts,

    CASE (n % 5)
        WHEN 0 THEN 'page_view'
        WHEN 1 THEN 'click'
        WHEN 2 THEN 'add_to_cart'
        WHEN 3 THEN 'checkout'
        ELSE 'recommendation_shown'
    END AS event_type,

    CASE (n % 5)

        WHEN 0 THEN jsonb_build_object(
            'url', '/product/' || ((n % 200) + 1),
            'referrer',
                (ARRAY['google','direct','email','fb'])
                [(n % 4) + 1],
            'device',
                (ARRAY['mobile','desktop','tablet'])
                [(n % 3) + 1]
        )

        WHEN 1 THEN jsonb_build_object(
            'element_id', 'btn_' || ((n % 50) + 1),
            'page_url', '/category/' || ((n % 20) + 1),
            'device',
                (ARRAY['mobile','desktop'])
                [(n % 2) + 1]
        )

        WHEN 2 THEN jsonb_build_object(
            'product_id', (n % 200) + 1,
            'price', ((n % 100) + 10)::NUMERIC(10, 2),
            'quantity', (n % 5) + 1,
            'tags', jsonb_build_array(
                (ARRAY['promo','new','clearance'])
                [(n % 3) + 1]
            )
        )

        WHEN 3 THEN jsonb_build_object(
            'order_id', 'o_' || (n + 100000),
            'total', ((n % 500) + 50)::NUMERIC(10, 2),
            'items_count', (n % 10) + 1,
            'payment_method',
                (ARRAY['card','paypal','crypto'])
                [(n % 3) + 1]
        )

        ELSE jsonb_build_object(
            'model_name', 'rec_v' || ((n % 5) + 1),
            'model_version', '1.' || (n % 10),
            'recommendations', jsonb_build_array(
                (n % 200) + 1,
                ((n + 1) % 200) + 1,
                ((n + 2) % 200) + 1
            ),
            'experiment', 'exp_' || ((n % 3) + 1)
        )

    END AS payload

FROM generate_series(1, 50000) AS n;
"""

with engine.begin() as conn:
    conn.execute(text(insert_query))
    conn.execute(text("ANALYZE t7_hw_events;"))

print("50,000 events inserted successfully!")

50,000 events inserted successfully!


In [20]:
pd.read_sql("""
    SELECT
        event_type,
        COUNT(*) AS n_events
    FROM t7_hw_events
    GROUP BY event_type
    ORDER BY event_type;
""", engine)

,event_type,n_events
0,add_to_cart,10000
1,checkout,10000
2,click,10000
3,page_view,10000
4,recommendation_shown,10000


In [21]:
pd.read_sql("""
    SELECT DISTINCT ON (event_type)
        event_type,
        payload
    FROM t7_hw_events
    ORDER BY event_type, event_id;
""", engine)

,event_type,payload
0,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quanti..."
1,checkout,"{'total': 53.0, 'order_id': 'o_100003', 'items..."
2,click,"{'device': 'desktop', 'page_url': '/category/2..."
3,page_view,"{'url': '/product/6', 'device': 'tablet', 'ref..."
4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."


In [22]:
query_containment = """
SELECT
    event_id,
    user_id,
    event_ts,
    payload

FROM t7_hw_events

WHERE event_type = 'checkout'
  AND payload @> jsonb_build_object(
      'payment_method', 'card'
  )

LIMIT 10;
"""

pd.read_sql(query_containment, engine)

,event_id,user_id,event_ts,payload
0,3,4,2024-03-29 01:00:00+00:00,"{'total': 53.0, 'order_id': 'o_100003', 'items..."
1,18,19,2024-03-14 01:00:00+00:00,"{'total': 68.0, 'order_id': 'o_100018', 'items..."
2,33,34,2024-02-28 01:00:00+00:00,"{'total': 83.0, 'order_id': 'o_100033', 'items..."
3,48,49,2024-02-13 01:00:00+00:00,"{'total': 98.0, 'order_id': 'o_100048', 'items..."
4,63,64,2024-03-29 01:00:00+00:00,"{'total': 113.0, 'order_id': 'o_100063', 'item..."
5,78,79,2024-03-14 01:00:00+00:00,"{'total': 128.0, 'order_id': 'o_100078', 'item..."
6,93,94,2024-02-28 01:00:00+00:00,"{'total': 143.0, 'order_id': 'o_100093', 'item..."
7,108,109,2024-02-13 01:00:00+00:00,"{'total': 158.0, 'order_id': 'o_100108', 'item..."
8,123,124,2024-03-29 01:00:00+00:00,"{'total': 173.0, 'order_id': 'o_100123', 'item..."
9,138,139,2024-03-14 01:00:00+00:00,"{'total': 188.0, 'order_id': 'o_100138', 'item..."


In [23]:
query_key = """
SELECT
    event_id,
    event_type,
    payload

FROM t7_hw_events

WHERE payload ? 'tags'

LIMIT 10;
"""

pd.read_sql(query_key, engine)

,event_id,event_type,payload
0,2,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quanti..."
1,7,add_to_cart,"{'tags': ['new'], 'price': 17.0, 'quantity': 3..."
2,12,add_to_cart,"{'tags': ['promo'], 'price': 22.0, 'quantity':..."
3,17,add_to_cart,"{'tags': ['clearance'], 'price': 27.0, 'quanti..."
4,22,add_to_cart,"{'tags': ['new'], 'price': 32.0, 'quantity': 3..."
5,27,add_to_cart,"{'tags': ['promo'], 'price': 37.0, 'quantity':..."
6,32,add_to_cart,"{'tags': ['clearance'], 'price': 42.0, 'quanti..."
7,37,add_to_cart,"{'tags': ['new'], 'price': 47.0, 'quantity': 3..."
8,42,add_to_cart,"{'tags': ['promo'], 'price': 52.0, 'quantity':..."
9,47,add_to_cart,"{'tags': ['clearance'], 'price': 57.0, 'quanti..."


In [24]:
query_keys_any = """
SELECT
    event_id,
    event_type,
    payload

FROM t7_hw_events

WHERE payload ?| ARRAY[
    'experiment',
    'model_version'
]

LIMIT 10;
"""

pd.read_sql(query_keys_any, engine)

,event_id,event_type,payload
0,4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
1,9,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
2,14,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
3,19,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
4,24,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
5,29,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
6,34,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
7,39,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
8,44,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
9,49,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."


In [25]:
query_jsonpath_recommendations = """
SELECT
    event_id,

    jsonb_path_query(
        payload,
        '$.recommendations[*]'
    ) AS recommended_pid

FROM t7_hw_events

WHERE event_type = 'recommendation_shown'

LIMIT 15;
"""

pd.read_sql(query_jsonpath_recommendations, engine)

,event_id,recommended_pid
0,4,5
1,4,6
2,4,7
3,9,10
4,9,11
5,9,12
6,14,15
7,14,16
8,14,17
9,19,20


In [26]:
query_jsonpath_checkout = """
SELECT
    event_id,

    payload ->> 'order_id' AS order_id,

    (payload ->> 'total')::NUMERIC(10, 2)
        AS total

FROM t7_hw_events

WHERE event_type = 'checkout'

  AND jsonb_path_exists(
      payload,
      '$.total ? (@ > 300)'
  )

LIMIT 10;
"""

pd.read_sql(query_jsonpath_checkout, engine)

,event_id,order_id,total
0,253,o_100253,303.0
1,258,o_100258,308.0
2,263,o_100263,313.0
3,268,o_100268,318.0
4,273,o_100273,323.0
5,278,o_100278,328.0
6,283,o_100283,333.0
7,288,o_100288,338.0
8,293,o_100293,343.0
9,298,o_100298,348.0


In [27]:
pd.read_sql("""
    SELECT
        COUNT(*) AS total_events,

        COUNT(DISTINCT event_type)
            AS total_event_types,

        COUNT(DISTINCT user_id)
            AS total_users,

        COUNT(*) FILTER (
            WHERE payload ? 'tags'
        ) AS events_with_tags,

        COUNT(*) FILTER (
            WHERE payload ? 'recommendations'
        ) AS events_with_recommendations

    FROM t7_hw_events;
""", engine)

,total_events,total_event_types,total_users,events_with_tags,events_with_recommendations
0,50000,5,1000,10000,10000


GIN-index demo + EXPLAIN ANALYZE

In [28]:
from sqlalchemy import text
import pandas as pd

explain_results = {}

def run_explain(label, query):
    with engine.connect() as conn:
        result = conn.execute(
            text("EXPLAIN (ANALYZE, BUFFERS) " + query)
        )
        plan = "\n".join(row[0] for row in result)

    explain_results[label] = plan

    print(f"\n--- {label} ---\n")
    print(plan)

    return plan

print("EXPLAIN helper is ready!")

EXPLAIN helper is ready!


In [29]:
with engine.begin() as conn:

    conn.execute(text("""
        DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops;
    """))

    conn.execute(text("""
        DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops;
    """))

    conn.execute(text("""
        DROP INDEX IF EXISTS idx_t7_events_device_btree;
    """))

    conn.execute(text("""
        ANALYZE t7_hw_events;
    """))

print("Indexes removed. Table analyzed.")

Indexes removed. Table analyzed.


In [30]:
plan_before_gin = run_explain(
    "1. Containment BEFORE GIN",

    """
    SELECT COUNT(*)
    FROM t7_hw_events
    WHERE payload @> jsonb_build_object(
        'payment_method', 'card'
    );
    """
)


--- 1. Containment BEFORE GIN ---

Aggregate  (cost=1761.49..1761.50 rows=1 width=8) (actual time=31.701..31.701 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=4595 width=0) (actual time=0.030..31.531 rows=3334 loops=1)
        Filter: (payload @> jsonb_build_object('payment_method', 'card'))
        Rows Removed by Filter: 46666
        Buffers: shared hit=1000
Planning:
  Buffers: shared hit=5
Planning Time: 0.572 ms
Execution Time: 31.778 ms


In [31]:
with engine.begin() as conn:

    conn.execute(text("""
        CREATE INDEX idx_t7_events_payload_gin_pathops
        ON t7_hw_events
        USING GIN (payload jsonb_path_ops);
    """))

    conn.execute(text("""
        ANALYZE t7_hw_events;
    """))

print("GIN index created successfully!")

GIN index created successfully!


In [32]:
pd.read_sql("""
    SELECT
        pg_size_pretty(
            pg_relation_size(
                'idx_t7_events_payload_gin_pathops'
            )
        ) AS gin_index_size;
""", engine)

,gin_index_size
0,1144 kB


In [33]:
plan_after_gin = run_explain(
    "2. Containment AFTER GIN",

    """
    SELECT COUNT(*)
    FROM t7_hw_events
    WHERE payload @> jsonb_build_object(
        'payment_method', 'card'
    );
    """
)


--- 2. Containment AFTER GIN ---

Aggregate  (cost=1065.35..1065.36 rows=1 width=8) (actual time=8.409..8.411 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=24.95..1059.58 rows=2309 width=0) (actual time=1.124..8.122 rows=3334 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('payment_method', 'card'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..24.37 rows=2309 width=0) (actual time=0.837..0.838 rows=3334 loops=1)
              Index Cond: (payload @> jsonb_build_object('payment_method', 'card'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=18
Planning Time: 0.702 ms
Execution Time: 8.494 ms


In [34]:
plan_referrer = run_explain(
    "3. Containment referrer = google",

    """
    SELECT COUNT(*)
    FROM t7_hw_events
    WHERE payload @> jsonb_build_object(
        'referrer', 'google'
    );
    """
)


--- 3. Containment referrer = google ---

Aggregate  (cost=1067.02..1067.03 rows=1 width=8) (actual time=3.500..3.501 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=25.33..1061.06 rows=2382 width=0) (actual time=0.357..3.381 rows=2500 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('referrer', 'google'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..24.74 rows=2382 width=0) (actual time=0.260..0.260 rows=2500 loops=1)
              Index Cond: (payload @> jsonb_build_object('referrer', 'google'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=1
Planning Time: 0.109 ms
Execution Time: 3.528 ms


In [35]:
plan_before_btree = run_explain(
    "4. Device equality BEFORE B-tree",

    """
    SELECT COUNT(*)
    FROM t7_hw_events
    WHERE payload ->> 'device' = 'mobile';
    """
)


--- 4. Device equality BEFORE B-tree ---

Aggregate  (cost=1750.62..1750.63 rows=1 width=8) (actual time=12.071..12.072 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=250 width=0) (actual time=0.058..11.483 rows=8333 loops=1)
        Filter: ((payload ->> 'device'::text) = 'mobile'::text)
        Rows Removed by Filter: 41667
        Buffers: shared hit=1000
Planning Time: 0.107 ms
Execution Time: 12.104 ms


In [36]:
with engine.begin() as conn:

    conn.execute(text("""
        CREATE INDEX idx_t7_events_device_btree
        ON t7_hw_events ((payload ->> 'device'));
    """))

    conn.execute(text("""
        ANALYZE t7_hw_events;
    """))

print("Expression B-tree index created!")

Expression B-tree index created!


In [37]:
plan_after_btree = run_explain(
    "5. Device equality AFTER B-tree",

    """
    SELECT COUNT(*)
    FROM t7_hw_events
    WHERE payload ->> 'device' = 'mobile';
    """
)


--- 5. Device equality AFTER B-tree ---

Aggregate  (cost=1243.33..1243.34 rows=1 width=8) (actual time=3.249..3.250 rows=1 loops=1)
  Buffers: shared hit=1 read=9
  ->  Bitmap Heap Scan on t7_hw_events  (cost=97.06..1222.43 rows=8358 width=0) (actual time=1.284..2.452 rows=8333 loops=1)
        Recheck Cond: ((payload ->> 'device'::text) = 'mobile'::text)
        Heap Blocks: exact=1000
        Buffers: shared hit=1 read=9
        ->  Bitmap Index Scan on idx_t7_events_device_btree  (cost=0.00..94.98 rows=8358 width=0) (actual time=0.878..0.878 rows=8333 loops=1)
              Index Cond: ((payload ->> 'device'::text) = 'mobile'::text)
              Buffers: shared read=9
Planning:
  Buffers: shared hit=18 read=1
Planning Time: 0.700 ms
Execution Time: 3.293 ms


In [38]:
import re

comparison = []

for label, plan in explain_results.items():

    match = re.search(
        r'Execution Time:\s*([\d.]+)\s*ms',
        plan
    )

    execution_time = (
        float(match.group(1)) if match else None
    )

    # Оператори сканування
    scan_types = [
        scan for scan in [
            'Seq Scan',
            'Bitmap Index Scan',
            'Bitmap Heap Scan',
            'Index Only Scan',
            'Index Scan'
        ]
        if scan in plan
    ]

    # -Планова вартість кореневого вузла
    cost_match = re.search(
        r'cost=([\d.]+)\.\.([\d.]+)',
        plan
    )

    total_cost = (
        float(cost_match.group(2))
        if cost_match else None
    )

    comparison.append({
        'query': label,
        'total_cost': total_cost,
        'execution_time_ms': execution_time,
        'scan_types': ', '.join(scan_types)
    })

df_comparison = pd.DataFrame(comparison)

display(df_comparison)

,query,total_cost,execution_time_ms,scan_types
0,1. Containment BEFORE GIN,1761.50,31.778,Seq Scan
1,2. Containment AFTER GIN,1065.36,8.494,"Bitmap Index Scan, Bitmap Heap Scan, Index Scan"
2,3. Containment referrer = google,1067.03,3.528,"Bitmap Index Scan, Bitmap Heap Scan, Index Scan"
3,4. Device equality BEFORE B-tree,1750.63,12.104,Seq Scan
4,5. Device equality AFTER B-tree,1243.34,3.293,"Bitmap Index Scan, Bitmap Heap Scan, Index Scan"


In [39]:
with open(
    "t7_hw_explain_results.txt",
    "w",
    encoding="utf-8"
) as f:

    for label, plan in explain_results.items():
        f.write(f"\n{'=' * 60}\n")
        f.write(f"{label}\n")
        f.write(f"{'=' * 60}\n")
        f.write(plan + "\n")

print("All EXPLAIN results saved successfully!")

All EXPLAIN results saved successfully!


JSONB vs Normalized + Reflection

In [40]:
with engine.begin() as conn:

    conn.execute(text("""
        DROP TABLE IF EXISTS t7_hw_checkouts_norm;
    """))

    conn.execute(text("""
        CREATE TABLE t7_hw_checkouts_norm (
            event_id INTEGER PRIMARY KEY,
            user_id INTEGER NOT NULL,
            event_ts TIMESTAMPTZ NOT NULL,
            order_id TEXT NOT NULL,
            total NUMERIC(10, 2) NOT NULL,
            items_count INTEGER NOT NULL
                CHECK (items_count > 0),
            payment_method TEXT NOT NULL
                CHECK (
                    payment_method IN (
                        'card', 'paypal', 'crypto'
                    )
                )
        );
    """))

print("Normalized checkout table created!")

Normalized checkout table created!


In [41]:
with engine.begin() as conn:

    conn.execute(text("""
        INSERT INTO t7_hw_checkouts_norm (
            event_id,
            user_id,
            event_ts,
            order_id,
            total,
            items_count,
            payment_method
        )

        SELECT
            event_id,
            user_id,
            event_ts,
            payload ->> 'order_id' AS order_id,
            (payload ->> 'total')::NUMERIC(10, 2)
                AS total,
            (payload ->> 'items_count')::INTEGER
                AS items_count,
            payload ->> 'payment_method'
                AS payment_method

        FROM t7_hw_events
        WHERE event_type = 'checkout';
    """))

    conn.execute(text("""
        CREATE INDEX idx_t7_checkouts_norm_pm
        ON t7_hw_checkouts_norm (payment_method);
    """))

    conn.execute(text("""
        ANALYZE t7_hw_checkouts_norm;
    """))

print("Checkout data inserted and index created!")

Checkout data inserted and index created!


In [42]:
pd.read_sql("""
    SELECT
        COUNT(*) AS n_checkouts,
        COUNT(DISTINCT payment_method) AS payment_methods,
        MIN(total) AS min_total,
        MAX(total) AS max_total
    FROM t7_hw_checkouts_norm;
""", engine)

,n_checkouts,payment_methods,min_total,max_total
0,10000,3,53.0,548.0


In [43]:
pd.read_sql("""
    SELECT *
    FROM t7_hw_checkouts_norm
    ORDER BY event_id
    LIMIT 10;
""", engine)

,event_id,user_id,event_ts,order_id,total,items_count,payment_method
0,3,4,2024-03-29 01:00:00+00:00,o_100003,53.0,4,card
1,8,9,2024-03-24 01:00:00+00:00,o_100008,58.0,9,crypto
2,13,14,2024-03-19 01:00:00+00:00,o_100013,63.0,4,paypal
3,18,19,2024-03-14 01:00:00+00:00,o_100018,68.0,9,card
4,23,24,2024-03-09 01:00:00+00:00,o_100023,73.0,4,crypto
5,28,29,2024-03-04 01:00:00+00:00,o_100028,78.0,9,paypal
6,33,34,2024-02-28 01:00:00+00:00,o_100033,83.0,4,card
7,38,39,2024-02-23 01:00:00+00:00,o_100038,88.0,9,crypto
8,43,44,2024-02-18 01:00:00+00:00,o_100043,93.0,4,paypal
9,48,49,2024-02-13 01:00:00+00:00,o_100048,98.0,9,card


In [44]:
plan_jsonb = run_explain(
    "6. JSONB checkout aggregation",

    """
    SELECT
        payload ->> 'payment_method'
            AS payment_method,

        AVG(
            (payload ->> 'total')::NUMERIC
        ) AS avg_total,

        COUNT(*) AS n

    FROM t7_hw_events

    WHERE event_type = 'checkout'

    GROUP BY payload ->> 'payment_method'

    ORDER BY avg_total DESC

    LIMIT 3;
    """
)


--- 6. JSONB checkout aggregation ---

Limit  (cost=1955.18..1955.19 rows=3 width=72) (actual time=14.517..14.518 rows=3 loops=1)
  Buffers: shared hit=1000
  ->  Sort  (cost=1955.18..1969.06 rows=5552 width=72) (actual time=14.515..14.516 rows=3 loops=1)
        Sort Key: (avg(((payload ->> 'total'::text))::numeric)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=1000
        ->  HashAggregate  (cost=1800.14..1883.42 rows=5552 width=72) (actual time=14.463..14.488 rows=3 loops=1)
              Group Key: (payload ->> 'payment_method'::text)
              Batches: 1  Memory Usage: 217kB
              Buffers: shared hit=1000
              ->  Seq Scan on t7_hw_events  (cost=0.00..1650.02 rows=10008 width=132) (actual time=0.023..8.738 rows=10000 loops=1)
                    Filter: (event_type = 'checkout'::text)
                    Rows Removed by Filter: 40000
                    Buffers: shared hit=1000
Planning:
  Buffers: shared hit=16
Planning Time

In [45]:
plan_normalized = run_explain(
    "7. Normalized checkout aggregation",

    """
    SELECT
        payment_method,

        AVG(total) AS avg_total,

        COUNT(*) AS n

    FROM t7_hw_checkouts_norm

    GROUP BY payment_method

    ORDER BY avg_total DESC

    LIMIT 3;
    """
)


--- 7. Normalized checkout aggregation ---

Limit  (cost=269.06..269.07 rows=3 width=46) (actual time=4.580..4.583 rows=3 loops=1)
  Buffers: shared hit=94
  ->  Sort  (cost=269.06..269.07 rows=3 width=46) (actual time=4.578..4.580 rows=3 loops=1)
        Sort Key: (avg(total)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=94
        ->  HashAggregate  (cost=269.00..269.04 rows=3 width=46) (actual time=4.558..4.562 rows=3 loops=1)
              Group Key: payment_method
              Batches: 1  Memory Usage: 24kB
              Buffers: shared hit=94
              ->  Seq Scan on t7_hw_checkouts_norm  (cost=0.00..194.00 rows=10000 width=11) (actual time=0.022..1.216 rows=10000 loops=1)
                    Buffers: shared hit=94
Planning Time: 0.184 ms
Execution Time: 4.644 ms


In [46]:
import re

def get_plan_metrics(plan):
    first_line = plan.splitlines()[0]

    cost_match = re.search(
        r'cost=[\d.]+\.\.([\d.]+)',
        first_line
    )

    time_match = re.search(
        r'Execution Time:\s*([\d.]+)\s*ms',
        plan
    )

    operators = [
        op for op in [
            'Seq Scan',
            'Bitmap Index Scan',
            'Bitmap Heap Scan',
            'Index Scan',
            'Index Only Scan',
            'HashAggregate',
            'GroupAggregate',
            'Sort'
        ]
        if op in plan
    ]

    return {
        'total_cost': (
            float(cost_match.group(1))
            if cost_match else None
        ),
        'execution_time_ms': (
            float(time_match.group(1))
            if time_match else None
        ),
        'operators': ', '.join(operators)
    }


comparison_5 = pd.DataFrame([
    {
        'version': 'JSONB',
        **get_plan_metrics(plan_jsonb)
    },
    {
        'version': 'Normalized',
        **get_plan_metrics(plan_normalized)
    }
])

display(comparison_5)

,version,total_cost,execution_time_ms,operators
0,JSONB,1955.19,14.622,"Seq Scan, HashAggregate, Sort"
1,Normalized,269.07,4.644,"Seq Scan, HashAggregate, Sort"


In [47]:
query_jsonb = """
SELECT
    payload ->> 'payment_method' AS payment_method,
    AVG((payload ->> 'total')::NUMERIC) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_events
WHERE event_type = 'checkout'
GROUP BY payload ->> 'payment_method'
ORDER BY avg_total DESC
LIMIT 3;
"""

query_norm = """
SELECT
    payment_method,
    AVG(total) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3;
"""

df_jsonb = pd.read_sql(query_jsonb, engine)
df_norm = pd.read_sql(query_norm, engine)

print("JSONB results:")
display(df_jsonb)

print("Normalized results:")
display(df_norm)

pd.testing.assert_frame_equal(
    df_jsonb.reset_index(drop=True),
    df_norm.reset_index(drop=True),
    check_dtype=False
)

print("Both queries return identical results!")

JSONB results:


,payment_method,avg_total,n
0,paypal,300.524752,3333
1,card,300.500000,3334
2,crypto,300.475248,3333


Normalized results:


,payment_method,avg_total,n
0,paypal,300.524752,3333
1,card,300.500000,3334
2,crypto,300.475248,3333


Both queries return identical results!


// MongoDB equivalent:
// Average checkout total per payment_method, top-3
// Pseudocode — not executed in PostgreSQL

db.events.aggregate([
  {
    $match: {
      event_type: "checkout"
    }
  },
  {
    $group: {
      _id: "$payload.payment_method",
      avg_total: {
        $avg: {
          $toDouble: "$payload.total"
        }
      },
      n: {
        $sum: 1
      }
    }
  },
  {
    $sort: {
      avg_total: -1
    }
  },
  {
    $limit: 3
  }
]);

In [48]:
labels = [
    "1. Containment BEFORE GIN",
    "2. Containment AFTER GIN",
    "4. Device equality BEFORE B-tree",
    "5. Device equality AFTER B-tree",
    "6. JSONB checkout aggregation",
    "7. Normalized checkout aggregation"
]

rows = []

for label in labels:
    plan = explain_results.get(label)

    if plan is not None:
        rows.append({
            'query': label,
            **get_plan_metrics(plan)
        })

df_reflection = pd.DataFrame(rows)
display(df_reflection)

,query,total_cost,execution_time_ms,operators
0,1. Containment BEFORE GIN,1761.50,31.778,Seq Scan
1,2. Containment AFTER GIN,1065.36,8.494,"Bitmap Index Scan, Bitmap Heap Scan, Index Scan"
2,4. Device equality BEFORE B-tree,1750.63,12.104,Seq Scan
3,5. Device equality AFTER B-tree,1243.34,3.293,"Bitmap Index Scan, Bitmap Heap Scan, Index Scan"
4,6. JSONB checkout aggregation,1955.19,14.622,"Seq Scan, HashAggregate, Sort"
5,7. Normalized checkout aggregation,269.07,4.644,"Seq Scan, HashAggregate, Sort"


## Reflection: JSONB vs Normalized Data and Index Performance

У цій роботі я дослідив вплив індексів на продуктивність JSONB-запитів у PostgreSQL та порівняв JSONB із нормалізованою моделлю зберігання даних
До створення GIN-індексу запит із оператором `@>` використовував `Seq Scan`, мав `total cost = 1761.50` і час виконання `31.778 ms` Після створення GIN-індексу план змінився на `Bitmap Index Scan` та `Bitmap Heap Scan`, Вартість знизилася до `1065.36`, а час — до `8.494 ms`- тобто запит прискорився приблизно у 3.74 раза.

GIN ефективний для containment операцій оскільки індексує вміст JSONB-документів. Проте він не оптимізує безпосередньо текстове порівняння `payload ->> 'device' = 'mobile'`, для цього був створений expression B-tree index. До індексування використовувався `Seq Scan` із часом `12.104 ms` і вартістю `1750.63`, після індексування з'явилися `Bitmap Index Scan` та `Bitmap Heap Scan` час зменшився до `3.293 ms` а вартість — до `1243.34`

GIN створює компроміс між читанням і записом: прискорює вибіркові пошукові запити, але збільшує використання дискового простору та накладні витрати на INSERT і UPDATE і DELETE

Порівняння моделей зберігання показало, що JSONB-агрегація виконувалася за `14.622 ms` із вартістю `1955.19` тоді як нормалізована - за `4.644 ms` із вартістю `269.07`, Обидва плани використовували `Seq Scan` `HashAggregate`  `Sort`. Нормалізована версія була приблизно в 3.15 раза швидшою хоча вона також обробляла меншу спеціалізовану таблицю

Для різнорідних ML event-log я обрав би JSONB, а для стабільних ML ознак - типізовані SQL таблиці. Новий тип події `refund` у JSONB потребував би нового payload та оновлення валідації. У нормалізованій моделі знадобилися б окрема таблиця `refunds` визначення типів обмежень цілісності та міграція ETL-процесів.